# 02 — Climatologia, tendências e mapas
Requer dados EAC4 reais processados. Não substitui dados ausentes por simulações. Gases permanecem em kg/kg; PM em µg/m³. A climatologia tem período de referência explícito.

In [ ]:
from pathlib import Path
from cams_air_quality.config import load_config
ROOT = Path.cwd() if (Path.cwd() / "config/config.yaml").exists() else Path.cwd().parent
cfg = load_config(ROOT / "config/config.yaml")


In [ ]:
from cams_air_quality.processing import open_database, product_root
if not (product_root(cfg) / "index.json").exists():
    raise RuntimeError("Base real ausente. Configure ADS e execute pilotos, download-monthly e consolidate.")
ds = open_database(cfg)
ds

In [ ]:
from cams_air_quality.extraction import area_mean
from cams_air_quality.statistics import annual_mean, climatology, anomalies, summaries
baseline_start, baseline_end = "2003-01", "2022-12"
pm25_global = area_mean(ds.pm25)
pm25_global.plot(figsize=(12, 4))

In [ ]:
climatology(pm25_global, baseline_start, baseline_end).plot(marker="o")

In [ ]:
from cams_air_quality.maps import plot_map
out = ROOT / "reports/notebook_maps"
out.mkdir(parents=True, exist_ok=True)
# Month, annual mean, monthly climatology, anomaly: no hidden spatial interpolation.
plot_map(ds.pm25.sel(time="2020-07").isel(time=0), out / "pm25_2020_07.png")
annual = annual_mean(ds.pm25)
plot_map(annual.sel(time="2020").isel(time=0), out / "pm25_2020_annual.png")
clim = climatology(ds.pm25, baseline_start, baseline_end)
plot_map(clim.sel(month=7), out / "pm25_july_climatology.png")
a = anomalies(ds.pm25, baseline_start, baseline_end)
plot_map(a.sel(time="2020-07").isel(time=0), out / "pm25_july_2020_anomaly.png", anomaly=True)

In [ ]:
# NO2 and O3: explicit retained mass mixing ratio units.
for name in ["no2", "o3"]:
    s = area_mean(ds[name])
    print(name, s.attrs["units"])
    s.plot(figsize=(12, 4))
    __import__("matplotlib.pyplot", fromlist=["show"]).show()
    climatology(s, baseline_start, baseline_end).plot(marker="o")
    __import__("matplotlib.pyplot", fromlist=["show"]).show()

In [ ]:
# Full first exploratory analysis requested in the project.
from cams_air_quality.analysis import explore
explore(cfg, ROOT / "reports/exploratory", baseline_start, baseline_end)
ds.close()

Médias globais são ponderadas por área, não por população. OLS em `summaries` é descritivo e não inclui inferência que ignore autocorrelação. Anos incompletos são excluídos da média anual.